# Oil Palm Disease Classification using ALOS-2 SAR

**Postgraduate Research** — Prerequisites & Environment Setup

This notebook sets up the full environment for processing ALOS-2 PALSAR-2 data on **Google Colab**:
- **ESA SNAP 13.0.0** (Sentinel Toolboxes) — SAR processing engine
- **esa_snappy** — Python bridge to SNAP's Java API
- **Scientific Python stack** — numpy, rasterio, scikit-learn, etc.

Persistent storage uses **Google Cloud Storage** (SNAP binary cached in a bucket to avoid re-downloading on each session).

---

## Prerequisites

### 0. Authenticate GCS & Set Project Root

**Before running**: create a GCS bucket at https://console.cloud.google.com/storage and
set `GCS_BUCKET` below to your bucket name.

SNAP (~1.5 GB) will be cached in `gs://<bucket>/snap/` so it only needs to be downloaded from ESA once.

In [ ]:
from google.colab import auth
auth.authenticate_user()

# --- CONFIGURE THESE ---
GCS_BUCKET = "sar-oilpalm"          # your GCS bucket name (no gs://)
PROJECT_ROOT = "/content/SAR-OilPalm"  # local working dir on Colab VM
# -----------------------

import os
os.makedirs(f"{PROJECT_ROOT}/data", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/snap", exist_ok=True)
os.chdir(PROJECT_ROOT)

print(f"GCS bucket: gs://{GCS_BUCKET}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {os.getcwd()}")

# Verify GCS access
!gsutil ls "gs://{GCS_BUCKET}" > /dev/null 2>&1 && echo "GCS access OK" || echo "GCS access FAILED — check bucket name and permissions"

GCS bucket: gs://sar-oilpalm
Project root: /content/SAR-OilPalm
Working directory: /content/SAR-OilPalm
GCS access OK


### 1. System Setup — Python Dependencies

Installs `uv`, then installs all Python packages to the Colab system Python via `uv pip install --system`.

In [ ]:
# System checks
import platform, sys, shutil, os
print(f"Platform: {platform.system()} {platform.release()}")
print(f"Architecture: {platform.machine()}")
print(f"Python: {sys.version}")
print(f"wget: {shutil.which('wget') or 'not found'}")
print(f"curl: {shutil.which('curl') or 'not found'}")
print(f"java: {shutil.which('java') or 'not found (SNAP bundles its own JRE)'}")

Platform: Linux 6.6.122+
Architecture: x86_64
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
wget: /usr/bin/wget
curl: /usr/bin/curl
java: /usr/bin/java


In [ ]:
%%bash
set -e

PROJECT_ROOT="/content/SAR-OilPalm"
cd "$PROJECT_ROOT"

echo "Installing uv..."
pip install uv -q
echo "uv: $(uv --version)"

echo "Installing packages to Colab system Python..."
uv pip install --system \
    esa-snappy \
    "numpy>=1.26" scipy \
    matplotlib seaborn \
    rasterio rioxarray \
    geopandas shapely pyproj \
    scikit-learn \
    jupyter ipykernel ipywidgets \
    tqdm joblib

echo ""
echo "=== Environment ready ==="
python --version

Installing uv...
uv: uv 0.11.28 (x86_64-unknown-linux-gnu)
Installing packages to Colab system Python...

=== Environment ready ===
Python 3.12.13


Using Python 3.12.13 environment at: /usr
Checked 16 packages in 93ms


### 2. Download & Install ESA SNAP 13.0.0

Restores SNAP from the GCS cache if available, otherwise downloads the installer from ESA,
installs it, and caches the result on GCS for future sessions.

In [ ]:
%%bash
set -e

PROJECT_ROOT="/content/SAR-OilPalm"
GCS_BUCKET="sar-oilpalm"
SNAP_HOME="$PROJECT_ROOT/snap"
INSTALLER="esa-snap_sentinel_linux-13.0.0.sh"
INSTALLER_URL="https://download.esa.int/step/snap/13.0/installers/${INSTALLER}"
GCS_SNAP_CACHE="gs://$GCS_BUCKET/snap-13.0.0.tar.gz"

# Check if SNAP is already installed locally
if [ -f "$SNAP_HOME/bin/snap" ]; then
    echo "SNAP already installed at $SNAP_HOME — skipping"
    ls -la "$SNAP_HOME/bin/" | head -5
    exit 0
fi

restore_from_cache() {
    echo "Restoring SNAP from GCS cache..."
    gsutil cp "$GCS_SNAP_CACHE" - | tar xzf - -C "$PROJECT_ROOT"
    echo "SNAP restored to $SNAP_HOME"
    return 0
}

# Try restoring from GCS cache first
if gsutil -q stat "$GCS_SNAP_CACHE" 2>/dev/null; then
    restore_from_cache
    exit 0
fi

echo "No GCS cache found — downloading from ESA..."

# Download installer (cache the installer on GCS too)
GCS_INSTALLER_CACHE="gs://$GCS_BUCKET/installers/$INSTALLER"
if gsutil -q stat "$GCS_INSTALLER_CACHE" 2>/dev/null; then
    echo "Downloading installer from GCS cache..."
    gsutil cp "$GCS_INSTALLER_CACHE" "$INSTALLER"
else
    echo "Downloading installer from ESA (~1 GB)..."
    wget "$INSTALLER_URL" -O "$INSTALLER"
    gsutil cp "$INSTALLER" "$GCS_INSTALLER_CACHE"
fi
chmod +x "$INSTALLER"
echo "Installer ready: $(du -h $INSTALLER | cut -f1)"

# Create response file for quiet headless install
cat > response.varfile << 'EOF'
sys.adminRights$Boolean=false
sys.component.RSTB$Boolean=true
sys.component.S1TBX$Boolean=true
sys.component.S2TBX$Boolean=true
sys.component.S3TBX$Boolean=false
sys.component.SNAP$Boolean=true
sys.installationDir=__SNAP_HOME__
sys.languageId=en
sys.programGroupDisabled$Boolean=true
createDesktopLinkAction$Boolean=false
executeLauncherWithPythonAction$Boolean=false
deleteSnapDir$Boolean=false
EOF
sed -i "s|__SNAP_HOME__|$SNAP_HOME|" response.varfile

# Run installer in quiet mode
mkdir -p "$SNAP_HOME"
echo "Installing SNAP to $SNAP_HOME (this may take a few minutes)..."
bash "$INSTALLER" -q -varfile response.varfile
echo "SNAP installation complete"

# Cache the installed SNAP on GCS for next session
echo "Caching SNAP installation on GCS..."
tar czf /tmp/snap-13.0.0.tar.gz -C "$PROJECT_ROOT" snap
gsutil cp /tmp/snap-13.0.0.tar.gz "$GCS_SNAP_CACHE"
rm -f /tmp/snap-13.0.0.tar.gz
echo "SNAP cached at $GCS_SNAP_CACHE"

ls -la "$SNAP_HOME/bin/" | head -5

SNAP already installed at /content/SAR-OilPalm/snap — skipping
total 104
drwxr-xr-x  2 root root  4096 Jul 15 08:32 .
drwxr-xr-x 12 root root  4096 Jul 15 08:32 ..
-rwxr-xr-x  1 root root 13800 Oct 29  2025 gpt
-rw-r--r--  1 root root   250 Jul 15 08:32 gpt.vmoptions


### 3. Configure esa_snappy (SNAP ↔ Python Bridge)

Connects SNAP's Java engine to the Colab system Python.

In [ ]:
%%bash
set -e

PROJECT_ROOT="/content/SAR-OilPalm"
SNAP_HOME="$PROJECT_ROOT/snap"

export PATH="$SNAP_HOME/bin:$PATH"

echo "=== Running snappy-conf ==="
"$SNAP_HOME/bin/snappy-conf" "$(which python)"

echo ""
echo "=== GPT CLI ==="
"$SNAP_HOME/bin/gpt" -h 2>&1 | head -8 || true

echo ""
echo "=== Setup Complete ==="

=== Running snappy-conf ===
Configuring ESA SNAP-Python interface...
Found esa_snappy installed in '/usr/local/lib/python3.12/dist-packages'
Starting configuration...
Configuration finished successful!
Done. The SNAP-Python interface is located in '/usr/local/lib/python3.12/dist-packages/esa_snappy'
The executable of the Python environment is located at '/usr/local/bin/python'

=== GPT CLI ===
INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.
Usage:
  gpt <op>|<graph-file> [options] [<source-file-1> <source-file-2> ...]

Description:
  This tool is used to execute SNAP raster data operators in batch-mode. The
  operators can be used stand-alone or combined as a directed acyclic graph

=== Setup Complete ===


OpenJDK 64-Bit Server VM warning: Options -Xverify:none and -noverify were deprecated in JDK 13 and will likely be removed in a future release.


In [ ]:
import esa_snappy
from esa_snappy import ProductIO, GPF
print('OK - esa_snappy is configured and ready')

OK - esa_snappy is configured and ready


### 4. Quick Test

Verifies the full pipeline — `esa_snappy` imports and SNAP GPT CLI.

In [ ]:
print('esa_snappy is ready for ALOS-2 SAR processing')
ops = GPF.getDefaultInstance().getOperatorSpiRegistry().getOperatorSpis()
print(f'Available operators: {ops.size()}')

esa_snappy is ready for ALOS-2 SAR processing
Available operators: 156


---
**Next steps:**
1. Upload your ALOS-2 PALSAR-2 scenes to `gs://<bucket>/data/`
2. On a new Colab session, re-run **Cell 0** (GCS auth) and **Cell 3** (SNAP restore) — SNAP is restored from the GCS cache, no re-download needed
3. Proceed to preprocessing: calibration, speckle filtering, terrain correction